# 00 · Run `bench validate` on real BRIGHT

**Task 2 of the next-steps guide.** Checks that the new package reproduces the 3 Nov Colab results
(`bright_event_summary.csv`). This is also the verification step for the AI-written code
(record the outcome in `AI_USE_LOG.md`).

Before the freeze, confirmatory events show only a schema check, not class counts.

In [ ]:
# 1. Get the code. The repo is private: add a GitHub token (read access to this repo) in
#    Colab -> Secrets (key icon on the left) with the name GH_TOKEN, and enable notebook access.
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
REPO = f"https://{GH}@github.com/ThinukaLehan/damagesar-bench.git"
!git clone -q $REPO /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
%cd /content/damagesar-bench
!pip install -q -e .
!bench status

In [ ]:
# Download BRIGHT SAR + labels from Hugging Face straight to /content (~3.4 GB, ~1 min).
# Copying from Drive fails (Errno 107), so always download fresh.
import os, zipfile
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # BRIGHT is not gated; the token only avoids rate limits

ROOT = "/content/bright"
os.makedirs(ROOT, exist_ok=True)
for name in ["post-event.zip", "target.zip"]:
    folder = os.path.join(ROOT, name.replace(".zip", ""))
    if os.path.isdir(folder) and len(os.listdir(folder)) > 4000:
        print("already there:", folder); continue
    p = hf_hub_download("Kullervo/BRIGHT", name, repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
    print("unzipped", name)
for sub in ["post-event", "target"]:
    print(sub, len(os.listdir(os.path.join(ROOT, sub))), "files")

In [ ]:
# 2. Validate (takes a few minutes: every tile is opened once)
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/damagesar/outputs/validate_v0.1.0"
!bench validate --root /content/bright --out $OUT
!bench audit --out $OUT

In [ ]:
# 3. Compare with the 3 Nov summary (tile counts per event should match exactly)
import pandas as pd
new = pd.read_csv(f"{OUT}/event_summary.csv").set_index("event")
old_path = "/content/drive/MyDrive/damagesar/data/bright_event_summary.csv"
old = pd.read_csv(old_path)
print("old columns:", list(old.columns))
# Adjust 'event' / 'n_tiles' below to the old file's column names if they differ.
ev_col = "event" if "event" in old.columns else old.columns[0]
tile_col = next((c for c in old.columns if "tile" in c.lower()), None)
cmp = new[["role", "n_tiles", "n_excluded_nodata"]].join(old.set_index(ev_col)[[tile_col]], how="outer")
cmp["match"] = cmp["n_tiles"] == cmp[tile_col]
display(cmp)
print("ALL MATCH" if cmp["match"].all() else "MISMATCH: check before trusting the package")